# 추출 결과의 품질을 지표로 측정합니다

**이 교안의 목표는 품질 지표를 계산하고, 낮은 점수에서 무엇을 확인하고 고칠지 설명하는 것입니다.**  
day37에서 만든 관계·타입 검사를 사용하되, 오늘은 그 결과를 수치로 해석합니다.

| 확인할 질문 | 지표 | 배우는 곳 |
|---|---|---|
| 정해 둔 관계와 개체 유형을 지켰나요? | **스키마 준수율** | 1절 |
| 근거 문구가 원문에 그대로 있나요? | **근거 원문 일치율** | 2절 |
| 근거가 실제로 그 관계를 뒷받침하나요? | **샘플 정밀도**(표본의 근거 적합률) | 3절 |
| 있어야 할 관계를 얼마나 맞혔나요? | **골드 완전일치 정밀도·재현율·F1** | 4~5절 |

**학습 목표**

- 각 지표의 질문·분자·분모를 설명하고 코드로 계산할 수 있습니다.  
- 오류 행과 원문을 대조해 지표가 낮은 이유와 기본 교정 방향을 설명할 수 있습니다.  
- 작은 정답 목록을 직접 만들고, 맞힘·잘못 뽑음·놓침을 구분할 수 있습니다.  
- 실제 추출 전체의 지표와 검토할 오류 목록을 저장할 수 있습니다.

한국어 도서관 기록의 대출·분류 두 관계로 원리를 익히고 단위 프로젝트 2의 의약품 저장 자료에 적용합니다.  
모든 실습은 모델 호출 없이 실행합니다.  
**01에서는 지표를 해석하고 교정 방향을 정합니다. 02에서는 실제 항목을 교정하고 전체 결과를 재평가합니다.**  
낮은 점수만으로 원인을 확정하지 않습니다. 아래 코드로 오류 행과 원문을 먼저 확인합니다.

<img src="images/quality_questions.png" width="1000" alt="같은 추출에 형식·인용·의미·정답 목록이라는 서로 다른 질문을 합니다.">

같은 추출에 형식·인용·의미·정답 목록이라는 서로 다른 질문을 합니다.

## 0. 대출과 분류가 함께 있는 도서관 기록을 사용합니다

아래는 **설명을 위해 만든 가상 원문**입니다.

> (1) 민수는 파이썬 입문과 SQL 첫걸음을 빌렸습니다.  
> (2) 지우는 통계 기초를 빌렸습니다.  
> (3) 파이썬 입문과 SQL 첫걸음은 프로그래밍 분야로 분류됩니다.  
> (4) 통계 기초는 통계 분야로 분류됩니다.  
> (5) 민수는 머신러닝 실습을 빌릴 예정입니다.  
> (6) 지우는 딥러닝 이해를 빌리지 않았습니다.  
> (7) 사서는 지우에게 데이터 시각화를 추천했습니다.

이번에 지킬 스키마 규칙은 두 가지입니다.

| 규칙 | 주어 유형 | 관계 | 목적어 유형 |
|---|---|---|---|
| 1. 대출 | 사람 | 빌림 | 책 |
| 2. 분류 | 책 | 분류됨 | 분야 |

**트리플은 자신의 관계에 해당하는 규칙을 만족하면 됩니다.** 두 규칙을 동시에 만족해야 하는 것은 아닙니다.  
이 스키마 검사는 관계와 개체 유형만 확인합니다. 원문 의미의 정답 여부는 별도로 판단합니다.

**정답에 포함하는 기준**: 실제로 빌렸다고 명시한 대출, 책이 속한다고 명시한 분야를 기록합니다.  
대출 예정과 대출 부정은 빌림 관계의 정답에서 제외합니다. 추천은 이번 허용 관계에 없습니다.  
나열된 책은 한 권씩 나누어 기록합니다. 분야 이름은 원문 그대로 사용합니다.  
파일 읽기·저장 함수를 준비한 뒤 검사할 데이터를 봅니다.

<img src="images/library_two_schemas.png" width="1000" alt="빌림은 사람→책, 분류됨은 책→분야입니다. 관계를 고른 뒤 양쪽 개체 유형을 확인합니다.">

빌림은 사람→책, 분류됨은 책→분야입니다. 관계를 고른 뒤 양쪽 개체 유형을 확인합니다.

In [ ]:
# 실습에 공통으로 쓸 파일 경로와 읽기, 저장 함수를 준비합니다.

import json
import random
from collections import Counter
from pathlib import Path

data_dir = Path("data")  # 제공된 원문, 추출된 트리플, 골드 파일이 있는 폴더입니다.
output_dir = Path("output")  # 직접 계산한 지표와 검토 기록을 저장할 폴더입니다.
output_dir.mkdir(exist_ok=True)

def load_rows(filename):
    """data 폴더의 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    return [json.loads(line) for line in (data_dir / filename).read_text(encoding="utf-8").splitlines() if line.strip()]

def write_json(filename, value):
    """이번 실습의 결과를 output 폴더에 JSON으로 저장합니다."""
    (output_dir / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_rows(filename, rows):
    """검토 기록을 한 줄에 한 항목인 JSONL로 저장합니다."""
    content = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    (output_dir / filename).write_text(content + "\n", encoding="utf-8")

def triple_key(row):
    """고유 관계를 비교할 (주어, 관계, 목적어) 튜플을 돌려줍니다."""

    # 평가 전에 표기를 바꾸지 않습니다. 이름 정규화는 다음 단원에서 배웁니다.
    return (row["subject"], row["relation"], row["object"])

In [ ]:
# 도서관 원문과 추출된 트리플을 준비하고 각 행의 타입, 근거를 확인합니다.

demo_sentences = [
    "민수는 파이썬 입문과 SQL 첫걸음을 빌렸습니다.",
    "지우는 통계 기초를 빌렸습니다.",
    "파이썬 입문과 SQL 첫걸음은 프로그래밍 분야로 분류됩니다.",
    "통계 기초는 통계 분야로 분류됩니다.",
    "민수는 머신러닝 실습을 빌릴 예정입니다.",
    "지우는 딥러닝 이해를 빌리지 않았습니다.",
    "사서는 지우에게 데이터 시각화를 추천했습니다.",
]
demo_text = " ".join(demo_sentences)
# subject와 object는 주어와 목적어, relation은 관계, *_type은 유형, evidence는 추출된 근거입니다.
demo_raw = [
    {"subject": "민수", "subject_type": "사람", "relation": "빌림", "object": "파이썬 입문",
     "object_type": "책", "evidence": demo_sentences[0]},
    {"subject": "민수", "subject_type": "사람", "relation": "빌림", "object": "머신러닝 실습",
     "object_type": "책", "evidence": demo_sentences[4]},
    {"subject": "파이썬 입문", "subject_type": "책", "relation": "분류됨", "object": "프로그래밍",
     "object_type": "분야", "evidence": demo_sentences[2]},
    {"subject": "통계 기초", "subject_type": "책", "relation": "분류됨", "object": "통계",
     "object_type": "분야", "evidence": "통계 기초는 통계 분야 책입니다."},
    {"subject": "지우", "subject_type": "사람", "relation": "빌림", "object": "딥러닝 이해",
     "object_type": "책", "evidence": demo_sentences[5]},
    {"subject": "사서", "subject_type": "사람", "relation": "추천함", "object": "데이터 시각화",
     "object_type": "책", "evidence": demo_sentences[6]},
    {"subject": "민수", "subject_type": "사람", "relation": "분류됨", "object": "프로그래밍",
     "object_type": "분야", "evidence": demo_sentences[2]},
    {"subject": "SQL 첫걸음", "subject_type": "책", "relation": "분류됨", "object": "지우",
     "object_type": "사람", "evidence": demo_sentences[2]},
]

for number, row in enumerate(demo_raw, 1):
    print(number, triple_key(row), "/ 유형:", row["subject_type"], "→", row["object_type"])
    print("근거:", row["evidence"])

## 1. 스키마 준수율은 정해 둔 형식을 지킨 비율입니다

**스키마**는 어떤 관계와 개체 유형으로 기록할지 정한 규칙입니다.  
지난 시간의 검사를 사용해 통과와 기각을 셉니다.

| 번호 | 추출된 트리플 | 관계·타입 검사 | 이유 |
|---|---|---|---|
| 1 | 민수 → 빌림 → 파이썬 입문 | 통과 | 사람 → 빌림 → 책 |
| 2 | 민수 → 빌림 → 머신러닝 실습 | 통과 | 사람 → 빌림 → 책 |
| 3 | 파이썬 입문 → 분류됨 → 프로그래밍 | 통과 | 책 → 분류됨 → 분야 |
| 4 | 통계 기초 → 분류됨 → 통계 | 통과 | 책 → 분류됨 → 분야 |
| 5 | 지우 → 빌림 → 딥러닝 이해 | 통과 | 사람 → 빌림 → 책 |
| 6 | 사서 → 추천함 → 데이터 시각화 | 기각 | 추천함은 허용 관계가 아님 |
| 7 | 민수 → 분류됨 → 프로그래밍 | 기각 | 분류됨의 주어는 책이어야 함 |
| 8 | SQL 첫걸음 → 분류됨 → 지우 | 기각 | 분류됨의 목적어는 분야여야 함 |

**스키마 준수율 = 통과 행 수 ÷ 검사한 전체 행 수**

여기서는 `5 / 8 = 62.50%`입니다. 기각한 행도 분모에 넣습니다.  
2번의 대출 예정과 5번의 대출 부정은 형식 검사만으로 걸러지지 않습니다.  
**허용 관계인지 먼저 확인하고, 그 관계의 주어·목적어 타입을 검사합니다.** 원문의 의미는 3절에서 별도로 판정합니다.

In [ ]:
# 두 관계의 스키마 준수율을 계산하고 기각 사유를 확인합니다.

# 한 트리플에는 그 관계에 해당하는 타입 규칙을 적용합니다.
demo_signatures = {"빌림": ("사람", "책"), "분류됨": ("책", "분야")}

def check_demo_schema(row):
    """트리플의 관계, 타입 위반 사유를 돌려주고 통과하면 None을 돌려줍니다."""
    if row["relation"] not in demo_signatures:
        return "허용 관계가 아님"
    subject_type, object_type = demo_signatures[row["relation"]]
    if row["subject_type"] != subject_type:
        return f"주어는 {subject_type} 타입이어야 함"
    if row["object_type"] != object_type:
        return f"목적어는 {object_type} 타입이어야 함"

    return None

# 의미를 검사하기 전 단계입니다. 원본을 고치지 않고 기각 목록에만 사유를 붙입니다.
# demo_predictions는 스키마 통과 목록입니다. 관계가 맞다고 판정된 정답 목록은 아닙니다.
demo_predictions, demo_rejected = [], []
for row in demo_raw:
    reason = check_demo_schema(row)
    if reason is None:
        demo_predictions.append(row)
    else:
        demo_rejected.append(dict(row, reject_reason=reason))
demo_schema_rate = len(demo_predictions) / len(demo_raw)

print(f"스키마 준수율: {len(demo_predictions)} / {len(demo_raw)} = {demo_schema_rate:.2%}")
for row in demo_rejected:
    print("기각:", triple_key(row), "/ 사유:", row["reject_reason"])

### ✅ 바로 확인 퀴즈

파이썬 입문 → 분류됨 → 프로그래밍은 빌림 규칙의 타입과 다릅니다. 스키마 위반인가요?

<details><summary>정답 보기</summary>

아닙니다. 분류됨에 해당하는 책→분야 규칙을 만족하므로 통과합니다. 해당 관계의 규칙을 적용합니다.

</details>

### 단위 프로젝트 2의 의약품 추출에 적용합니다

실제 실습은 의약품 제품 문서 8개를 사용합니다.  
`fa_triples.jsonl`의 통과 140행과 `fa_rejected.jsonl`의 기각 9행을 합쳐 다시 검사합니다.  
**기각 행도 분모에 포함해야 원래 검사 대상의 준수율을 구할 수 있습니다.**

| 관계 | 주어 타입 | 허용 목적어 타입 |
|---|---|---|
| CONTAINS (성분 함유) | Drug (의약품) | Ingredient (성분) |
| TREATS (치료 대상) | Drug | Symptom (증상) |
| HAS_SIDE_EFFECT (이상반응) | Drug | Symptom |
| CAUTION_FOR (주의 대상) | Drug | RiskGroup (위험군) |
| INTERACTS_WITH (상호작용) | Drug | Drug 또는 Ingredient |

이 규칙은 단위 프로젝트 2의 `version_c_drugs/solution/ontology.py`를 따릅니다.  
**여기서는 관계와 타입만 검사합니다.** 근거 인용과 관계의 의미는 다음 단계에서 확인합니다.  
JSON 문법과 필수 필드 검사는 이미 끝나 읽을 수 있는 저장 자료입니다.

In [ ]:
# 단위 프로젝트 2의 의약품 관계와 타입 규칙을 검사합니다.

# 값은 주어 타입과 허용 목적어 타입 목록입니다. 상호작용은 약과 성분 모두 허용합니다.
signatures = {
    "CONTAINS": ("Drug", ("Ingredient",)),
    "TREATS": ("Drug", ("Symptom",)),
    "HAS_SIDE_EFFECT": ("Drug", ("Symptom",)),
    "CAUTION_FOR": ("Drug", ("RiskGroup",)),
    "INTERACTS_WITH": ("Drug", ("Drug", "Ingredient")),
}

def check_signature(row):
    """관계와 타입의 위반 사유를 돌려주고 통과하면 None을 돌려줍니다."""
    if row["relation"] not in signatures:
        return "허용 관계가 아님"
    subject_type, object_types = signatures[row["relation"]]
    if row["subject_type"] != subject_type:
        return f"주어 타입은 {subject_type}이어야 함"
    # 허용 타입이 두 개인 관계도 있으므로 == 대신 in으로 확인합니다.
    if row["object_type"] not in object_types:
        return f"목적어 타입은 {', '.join(object_types)} 중 하나여야 함"
    return None

def split_schema(rows):
    """추출을 스키마 통과 목록과 사유가 붙은 기각 목록으로 나눕니다."""
    valid, rejected = [], []
    for row in rows:
        reason = check_signature(row)
        if reason is None:
            valid.append(row)
        else:
            # 원본을 고치지 않고 이번 검사에서 확인한 사유만 덧붙입니다.
            rejected.append(dict(row, reject_reason=reason))
    return valid, rejected

In [ ]:
# 제품 8개의 원문과 통과, 기각 결과를 읽어 다시 검사할 목록을 만듭니다.

# fa_corpus.jsonl: 제품명, 전체 원문, 효능과 이상반응 절이 담긴 제품 8개의 문서입니다.
fa_docs = {row["doc_id"]: row for row in load_rows("fa_corpus.jsonl")}

# fa_triples.jsonl: 원본 프로젝트의 스키마, 근거, 절 검사를 통과한 140행입니다.
fa_saved = load_rows("fa_triples.jsonl")

# fa_rejected.jsonl: 같은 제품에서 기각된 9행과 당시의 기각 사유입니다.
fa_previous_rejected = load_rows("fa_rejected.jsonl")

# 당시 통과 여부와 관계없이 두 목록을 합칩니다. 이번 검사 결과는 별도 변수에 담습니다.
fa_raw = fa_saved + fa_previous_rejected
fa_scoped = [row for row in fa_raw if row["source_doc_id"] in fa_docs]

print("문서 수:", len(fa_docs), "/ 검사할 행 수:", len(fa_scoped))
print("첫 트리플:", triple_key(fa_scoped[0]))

### 스키마 준수율이 낮으면 LLM의 출력 제약을 보완합니다

**LLM이 온톨로지에서 정한 관계나 타입 규칙을 어기면 스키마 준수율이 낮아집니다.**

| LLM이 잘못 출력한 것 | 교정 방향 |
|---|---|
| 허용 목록에 없는 관계나 타입 | 구조화된 출력의 `Literal` 또는 `enum`으로 선택 가능한 값을 제한 |
| 관계에 맞지 않는 타입 조합 | 관계별 타입 조합도 제한. 예: `CONTAINS`는 `Drug → Ingredient` |
| 개체의 유형을 잘못 판단 | 타입의 뜻과 올바른 예시를 프롬프트에 추가하고 다시 추출 |

구조화된 출력은 허용하지 않은 값이 나오는 문제를 줄입니다.  
**허용 타입 이름만 제한하면 잘못된 조합은 남을 수 있으므로 관계별 검사도 필요합니다.**  
출력 제약을 고친 뒤 같은 규칙으로 다시 검사합니다.

### 🖐️ 함께 따라하기: 의약품 추출의 스키마 준수율을 계산합니다

**할 일**  
- `split_schema(fa_scoped)`의 결과를 **fa_valid**, **fa_schema_rejected**에 담으세요.  
- **my_schema_rate**에 통과 수를 전체 검사 수로 나눈 값을 담고 출력하세요.  
- 이번 기각 사유를 `Counter`로 세어 **my_schema_reasons**에 담고 출력하세요.

**확인 기준**: 149행 중 149행 통과, 스키마 기각은 0행입니다.  
기존 기각 파일의 9행은 근거나 절 검사에서 제외된 행이므로 스키마 위반으로 세지 않습니다.

In [ ]:
# 🖐️ 함께 따라하기: 의약품 추출의 스키마 준수율을 계산합니다

# 통과 파일과 기각 파일을 합친 fa_scoped 전체를 검사하세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

기존 기각 파일에 9행이 있으므로 스키마 기각도 9행인가요?

<details><summary>정답 보기</summary>

아닙니다. 이 9행은 근거나 절 검사에서 기각됐습니다. 관계와 타입만 검사하면 149행 모두 통과합니다.

</details>

## 2. 근거 원문 일치율은 인용문을 검사합니다

**근거 원문 일치율 = 원문에서 그대로 찾은 근거 행 수 ÷ 근거를 검사한 행 수**

추출의 evidence가 출처 문서에 **문자열 그대로 포함되는지** 확인합니다.  
이번 검사의 대상은 스키마 통과 행 전체입니다.  
중복 관계도 근거가 다를 수 있으므로 행마다 셉니다.

| 검사 결과 | 뜻 |
|---|---|
| 근거 없음 | 빈 문자열 또는 공백뿐임 |
| 원문 일치 | 비어 있지 않은 근거가 출처 원문에 그대로 있음 |
| 원문 확인 필요 | 요약·공백 변경·줄임표 등으로 그대로 찾을 수 없음 |

스키마를 통과한 5행 중 4행의 근거는 원문과 일치합니다.  
통계 기초의 분류 근거는 “통계 분야 책입니다”로 바뀌어 원문 그대로의 인용이 아닙니다.  
반대로 대출 예정·대출 부정 문장은 정확히 인용했어도 빌림 관계를 뒷받침하지 않습니다.  
**인용이 맞는지와 관계의 의미가 맞는지는 다른 질문입니다.**  
원문과 일치한 행은 **사용 후보**, 나머지는 **인용 교정 목록**으로 분리합니다.  
3절의 표본은 사용 후보에서 뽑습니다. 교정 목록은 지우지 않고 근거를 고친 뒤 다시 검사합니다.

In [ ]:
# 스키마 통과 행의 근거가 원문에 그대로 있는지 확인하고 일치율을 계산합니다.

# 빈 문자열은 어떤 문자열에도 포함되므로 먼저 제외합니다.
demo_verbatim = [row for row in demo_predictions
                 if row["evidence"].strip() and row["evidence"] in demo_text]
demo_quote_review = [row for row in demo_predictions if row not in demo_verbatim]
demo_verbatim_rate = len(demo_verbatim) / len(demo_predictions)
print(f"근거 원문 일치율: {len(demo_verbatim)} / {len(demo_predictions)} = {demo_verbatim_rate:.2%}")

for row in demo_predictions:
    matches = bool(row["evidence"].strip()) and row["evidence"] in demo_text
    print(triple_key(row), "/", "원문 일치" if matches else "원문 확인 필요")

print("원문 그대로지만 의미가 틀린 관계의 근거:", demo_predictions[1]["evidence"])

In [ ]:
# 의약품 추출의 근거를 '없음, 원문 일치, 확인 필요'로 분류할 함수를 만듭니다.

def evidence_status(row):
    """의약품 추출의 근거가 비었는지, 원문에 있는지, 확인이 필요한지 돌려줍니다."""
    evidence = row["evidence"]
    if not evidence.strip():
        return "근거 없음"
    if evidence in fa_docs[row["source_doc_id"]]["text"]:
        return "원문 일치"

    return "원문 확인 필요"

print("첫 추출의 근거:", fa_scoped[0]["evidence"])
print("인용 검사:", evidence_status(fa_scoped[0]))

### 🖐️ 함께 따라하기: 의약품 추출의 근거 원문 일치율을 계산합니다

**할 일**  
- **fa_valid** 전체에 evidence_status를 적용해 **my_evidence_counts**에 상태별 개수를 담으세요.  
- **my_verbatim_rate**에 원문 일치 수를 검사한 행 수로 나눈 값을 담으세요.  
- 두 변수를 출력하고, 원문 일치 행은 **fa_candidates**, 나머지는 **fa_quote_review**에 담으세요.

**확인 기준**: 149행 중 146행의 근거가 원문과 일치하고, 3행은 인용 교정 대상입니다. 분모는 근거를 검사한 149행입니다.

In [ ]:
# 🖐️ 함께 따라하기: 의약품 추출의 근거 원문 일치율을 계산합니다

# 상태별 개수를 세고 원문 일치 수를 검사한 전체 행 수로 나누세요.
# 여기에 코드를 작성하세요.

### 근거 원문 일치율이 낮으면 LLM의 인용 방식과 출처 연결을 고칩니다

**LLM이 근거를 빠뜨리거나 원문을 요약해서 쓰면 근거 원문 일치율이 낮아집니다.**  
다른 문서와 잘못 연결해도 일치하지 않으므로 출처 ID도 확인합니다.

| 확인한 오류 | 교정 방향 |
|---|---|
| 근거를 비워 둠 | 구조화된 출력에 `evidence`를 필수로 지정하고, 빈 값이면 근거를 포함해 다시 추출 |
| 원문을 요약하거나 공백과 문장부호를 바꿈 | “**근거는 원문의 연속된 구절을 수정 없이 복사하라**”고 지시하고 문자열 포함 여부를 검사 |
| 다른 문서의 근거가 연결됨 | 입력 문서 ID를 코드에서 추출 결과에 붙이고, 해당 문서의 원문과 대조 |

**구조화된 출력으로 근거 필드를 요구할 수 있지만, 원문 그대로인지는 별도 문자열 검사가 필요합니다.**  
불일치한 행은 원문에서 관계를 확인한 뒤 인용을 교정하거나 다시 추출합니다.  
아래의 “통계 기초는 통계 분야 책입니다.”는 의미는 같아도 원문을 바꿔 쓴 인용입니다.

In [ ]:
# 같은 의미라도 인용 표현이 바뀌면 문자열 검사가 실패하는지 확인합니다.

quoted_text = demo_predictions[3]["evidence"]
print("현재 인용:", quoted_text)
print("원문에 그대로 있음:", quoted_text in demo_text)

source_quote = demo_sentences[3]
print("대조할 원문:", source_quote)
print("원문에 그대로 있음:", source_quote in demo_text)

### ✅ 바로 확인 퀴즈

근거 원문 일치율이 100%이면 추출 관계의 의미도 전부 맞나요?

<details><summary>정답 보기</summary>

아닙니다. '빌릴 예정'이라는 원문을 정확히 인용하면서 '빌림'으로 잘못 추출할 수 있습니다.

</details>

## 3. 샘플 정밀도는 사용할 트리플 중 맞는 관계의 비율을 추정합니다

**스키마와 근거 원문 검사를 모두 통과한 목록에서 표본을 뽑아 관계의 의미를 사람이 판정합니다.**  
원문을 그대로 인용했어도 “빌리지 않았다”를 `빌림`으로 추출할 수 있기 때문입니다.

1. 스키마 검사: 허용 관계와 타입인지 확인합니다.  
2. 근거 원문 검사: 인용 문자열이 출처 원문에 그대로 있는지 확인합니다.  
3. 표본 선정: 두 검사를 통과한 사용 후보에서 무작위로 일부를 뽑습니다.  
4. 사람 판정: 원문과 포함 기준을 읽고 트리플이 맞으면 1점, 틀리면 0점을 줍니다.

**샘플 정밀도 = 1점인 표본 수 ÷ 판정을 모두 마친 표본 수**

이는 **검사를 통과한 결과를 실제로 사용할 때의 품질**을 평가하는 방법입니다.  
실무에서도 최종 사용 결과를 평가할 때 적용할 수 있습니다. 모든 추출 평가에 의무인 순서는 아닙니다.  
원래 LLM 출력의 품질을 평가하려면 검사로 행을 제외하기 전의 전체 추출에서 표본을 뽑아야 합니다.

### 도서관의 사용 후보 4건을 판정합니다

전체 8건 중 스키마 통과는 5건, 그중 근거 원문 일치까지 통과한 것은 **4건**이며 `demo_verbatim`에 담겨 있습니다.  
`통계 기초 → 분류됨 → 통계`는 인용 문구가 달라 교정 목록으로 빠졌습니다. 이 표본에서 0점으로 세지 않습니다.

**평가 대상은 트리플입니다.** 주어와 목적어가 맞게 연결되고, 관계가 원문과 포함 기준에 맞는지 봅니다.  
완료된 대출과 명시된 분야 분류만 포함하며, 예정과 부정은 대출로 인정하지 않습니다.

| 추출된 트리플 | 추출된 근거 (`evidence`) | 판정과 이유 |
|---|---|---|
| 민수 → 빌림 → 파이썬 입문 | 민수는 파이썬 입문과 SQL 첫걸음을 빌렸습니다. | 1: 해당 책을 실제로 빌림 |
| 민수 → 빌림 → 머신러닝 실습 | 민수는 머신러닝 실습을 빌릴 예정입니다. | 0: 예정은 완료한 대출이 아님 |
| 파이썬 입문 → 분류됨 → 프로그래밍 | 파이썬 입문과 SQL 첫걸음은 프로그래밍 분야로 분류됩니다. | 1: 해당 책의 분야를 명시함 |
| 지우 → 빌림 → 딥러닝 이해 | 지우는 딥러닝 이해를 빌리지 않았습니다. | 0: 부정을 반대로 추출함 |

4건 전체를 읽으면 `2 / 4 = 50%`입니다. 아래에서는 3건만 뽑아 표본의 점수를 구합니다.  
**근거 문자열이 존재하는지만으로 1점을 주지 않습니다.** 이미 끝낸 스키마와 인용 검사를 다시 합산하는 점수도 아닙니다.  
미판정은 0점이 아닙니다. 정한 표본의 판정을 모두 마친 뒤 계산합니다.

In [ ]:
# 두 검사를 통과한 사용 후보에서 표본을 뽑고 위 판정표로 점수를 계산합니다.

# evidence_scores는 위 표의 관계 판정입니다. 코드가 문장을 이해해 자동 채점한 값이 아닙니다.
evidence_scores = {
    ("민수", "빌림", "파이썬 입문"): 1,
    ("민수", "빌림", "머신러닝 실습"): 0,
    ("파이썬 입문", "분류됨", "프로그래밍"): 1,
    ("지우", "빌림", "딥러닝 이해"): 0,
}

def support_rate(rows, score_table):
    """모든 항목의 판정이 끝났을 때만 표본의 정밀도를 돌려줍니다."""
    if not rows:
        return None
    missing = [triple_key(row) for row in rows if triple_key(row) not in score_table]
    if missing:
        raise ValueError(f"먼저 원문을 읽고 채점하세요: {missing}")
    scores = [score_table[triple_key(row)] for row in rows]
    if any(score not in (0, 1) for score in scores):
        raise ValueError("채점값은 0 또는 1이어야 합니다")
    return sum(scores) / len(scores)

# 같은 결과를 재현하도록 시드를 고정합니다. 5건이 아닌 두 검사 통과 4건에서 뽑습니다.
sample = random.Random(42).sample(demo_verbatim, 3)
for row in sample:
    print("뽑힌 트리플:", triple_key(row))
    print("추출된 근거:", row["evidence"])
    print("관계 판정:", evidence_scores[triple_key(row)])
    print()

sample_precision = support_rate(sample, evidence_scores)
sample_correct = sum(evidence_scores[triple_key(row)] for row in sample)
print(f"샘플 정밀도: {sample_correct} / {len(sample)} = {sample_precision:.2%}")

all_candidate_precision = support_rate(demo_verbatim, evidence_scores)
print(f"사용 후보 4건 전체의 정밀도: 2 / 4 = {all_candidate_precision:.2%}")

### 샘플 정밀도가 낮으면 0점 항목의 판정 이유를 읽습니다

원문과 대조해 **LLM이 어떤 내용을 잘못된 관계로 추출했는지** 확인합니다.  
도서관 예시에서는 “빌릴 예정”과 “빌리지 않았다”를 모두 `빌림`으로 잘못 추출했습니다.  
“**완료된 대출만 포함하고 예정과 부정은 제외하라**”는 지시와 제외 예시를 추가한 뒤 다시 추출합니다.  
구조화된 출력으로 형식을 맞춰도 이런 의미 오류는 남을 수 있으므로, 같은 기준으로 다시 판정합니다.

In [ ]:
# 의미가 틀린 두 항목을 골라 어떤 문맥을 잘못 읽었는지 확인합니다.

semantic_errors = [row for row in demo_verbatim if evidence_scores[triple_key(row)] == 0]
print("의미 오류 수:", len(semantic_errors))

for row in semantic_errors:
    print("추출된 트리플:", triple_key(row))
    print("대조할 근거:", row["evidence"])

실제 적용에서는 표본을 고르고 **표본을 뽑은 단계, 크기, 선정 방법, 원문, 판정 기준과 이유**를 남깁니다.  
작은 표본의 결과는 선택한 항목에 따라 달라집니다. 위 3건 표본의 비율과 사용 후보 4건 전체의 비율도 다릅니다.

### 같은 제품 8개의 사용 후보에서 검토 표본을 만듭니다

앞에서 만든 `fa_candidates`는 두 검사를 통과한 146행입니다.  
여기서 10행을 무작위로 뽑고 원문과 함께 저장합니다. **아직 채점하지 않았으므로 점수는 `None`입니다.**  
판정할 때는 제품명과 해당 절을 함께 읽습니다. 효능은 `TREATS`, 이상반응은 `HAS_SIDE_EFFECT`의 근거입니다.

In [ ]:
# 같은 8문서의 사용 후보에서 검토할 10행을 뽑습니다. 탈락한 행은 표본에 들어오지 않습니다.

fa_sample = random.Random(42).sample(fa_candidates, min(10, len(fa_candidates)))
fa_review_sheet = []
for row in fa_sample:
    fa_review_sheet.append({
        **row,
        "source_text": fa_docs[row["source_doc_id"]]["text"],
        "score": None,  # 원문과 기준을 읽고 판정을 마친 뒤 0 또는 1로 기록합니다.
        "note": "",
    })

# my_sample_review.jsonl: 직접 판정할 트리플, 근거, 전체 원문과 빈 점수 칸을 담은 채점표입니다.
write_rows("my_sample_review.jsonl", fa_review_sheet)
print("표본 선정 단계: 스키마와 근거 원문 검사 통과 후")
print("사용 후보:", len(fa_candidates), "/ 검토 표본:", len(fa_review_sheet))
print("샘플 정밀도: 미산출, 10건의 판정이 끝나면 계산합니다.")

### 판정이 기록된 의약품 표본으로 계산을 연습합니다

방금 만든 10건은 직접 검토할 채점표입니다.  
아래 계산에는 같은 단위 프로젝트의 **별도 47문서, 표본 50건**에 기록된 판정을 사용합니다.  
앞의 8문서 점수와 섞지 않습니다.


이 표본은 원래 프로젝트에서 스키마와 근거 검사를 통과한 저장본에서 왔습니다.  
**판정 예시는 원문을 대조한 수업용 자료입니다. 사람이 검증한 점수는 아니므로 원문과 이유를 직접 확인하세요.**  
기존 채점표의 일부 메모가 실제 인용과 달라 재검토했으며, 이전 판정도 새 파일에 남겼습니다.

**주어가 원문 제품명과 같고 해당 절이 관계를 뒷받침하면 1점, 검토 후 기준에 맞지 않으면 0점입니다.**  
0점은 현실에서 거짓이라는 단정이 아닙니다. 아직 검토하지 못한 항목은 미판정입니다.

| 관계 | 1점으로 판정할 원문 근거 |
|---|---|
| TREATS | 해당 제품의 [효능] 절에 목적어의 증상이나 질환을 사용 대상으로 명시 |
| HAS_SIDE_EFFECT | 해당 제품의 [이상반응] 절에 목적어의 반응을 명시 |
| INTERACTS_WITH | 해당 제품의 [상호작용] 절에 목적어 약물과 병용할 때의 금지, 주의나 상담을 명시 |
| CAUTION_FOR | 해당 제품의 [경고·주의] 절에 목적어의 대상에게 사용 금지, 주의나 상담을 명시 |
| CONTAINS | 제품명 괄호나 본문에 함유 성분을 명시. 주의문에 성분 이름이 있다는 이유만으로 인정하지 않음 |

문맥에서 뜻이 같은 약어와 괄호 풀이는 허용합니다. 5절의 문자열 완전일치와는 다른 판정입니다.  
이 표본은 뒤의 의약품 골드 실습 8문서와 대상이 다르므로 점수를 합치지 않습니다.

In [ ]:
# 판정표와 원문을 함께 읽어 어떤 제품과 문맥을 검토했는지 확인합니다.

# fa_spotcheck_reviewed.jsonl: 기존 추출 50건에 원문 대조 판정, 이유와 이전 판정을 붙인 자료입니다.
spot_rows = load_rows("fa_spotcheck_reviewed.jsonl")

# fa_spotcheck_corpus.jsonl: 이 표본에 해당하는 제품 47개의 원문과 절별 내용입니다.
spot_docs = {row["doc_id"]: row for row in load_rows("fa_spotcheck_corpus.jsonl")}

print("검토 표본:", len(spot_rows), "/ 출처 문서:", len({row["doc_id"] for row in spot_rows}))
print("판정 자료: 수업용 원문 대조 판정 예시")

# 제품명은 원문의 [제품명]에서, 관계의 의미는 해당 절에서 확인합니다.
# source_section은 대조한 원문의 절 이름, source_quote는 그 절에서 가져온 원문입니다.
for doc_id in ["drug_197000053", "drug_197100059", "drug_199502575"]:
    row = next(item for item in spot_rows if item["doc_id"] == doc_id)
    print("\n문서:", doc_id, "/ 제품명:", spot_docs[doc_id]["title"])
    print("추출된 트리플:", triple_key(row))
    print("추출된 근거:", row["evidence"])
    print("대조할 절:", row["source_section"])
    print("대조할 원문:", row["source_quote"])
    print("판정:", row["score"], "/ 이유:", row["note"])

**원문에 `(주어, 관계, 목적어)`가 그대로 쓰여 있어야 하는 것은 아닙니다.**  
제품명은 문서의 [제품명]으로 연결하고, “이 약”은 그 제품을 가리킵니다.  
어떤 관계인지는 문장과 절의 의미로 판단합니다.

| 추출된 트리플 | 원문에서 확인할 연결 | 이번 판정 |
|---|---|---|
| 액티피드정 → INTERACTS_WITH → 구아네티딘 | 액티피드정 문서의 [상호작용] 절에 구아네티딘과 병용 금지 문장이 있음 | 1 |
| 쎄레스톤지크림 → HAS_SIDE_EFFECT → 전염성 농가진 | 쎄레스톤지크림 문서의 [이상반응] 절에 해당 반응이 있음 | 1 |
| 삼익이부프로펜정 → CONTAINS → 이부프로펜 | 인용은 고용량 복용에 관한 주의문임. 성분 함유를 명시한 문장은 확인되지 않음 | 0 |

쎄레스톤지크림은 잘린 인용만 읽으면 목록의 의미가 불분명하지만, 원문의 절까지 읽으면 관계를 확인할 수 있습니다.  
따라서 원문을 참조하는 이번 관계 판정은 1점이며, **인용에 문맥을 더 남길 필요는 별도 문제**입니다.

### 🖐️ 함께 따라하기: 실제 추출 표본의 샘플 정밀도를 구합니다

**할 일**  
- spot_rows의 score를 합한 값을 **spot_correct**에 담으세요.  
- **spot_precision**에 적합 수를 표본 수로 나눈 값을 담고 분자·분모·백분율을 출력하세요.  
- 0점을 받은 모든 항목의 트리플, evidence와 note를 출력해 판정 이유를 읽으세요.

**확인 기준**: 표본은 50건이고 판정값은 모두 0 또는 1입니다.  
출력 이름은 '별도 의약품 표본의 근거 적합률'로 적습니다.

In [ ]:
# 🖐️ 함께 따라하기: 실제 추출 표본의 샘플 정밀도를 구합니다

# score의 합은 적합 수이고 분모는 이 채점표의 행 수입니다.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

표본 10건 중 8건만 채점했고 모두 적합했습니다. 최종 정밀도를 100%로 보고해도 될까요?

<details><summary>정답 보기</summary>

아직 안 됩니다. 남은 2건은 미판정입니다. 지금은 8/10건 검토 완료로 기록하고 전체 표본의 판정을 마친 뒤 계산합니다.

</details>

## 4. 놓친 관계를 찾으려면 골드셋이 필요합니다

표본 검토는 **뽑힌 것**만 읽습니다. 원문에 있는데 뽑지 못한 것도 세려면  
원문을 읽고 **있어야 할 관계의 목록**을 만들어야 합니다. 이 목록이 **골드셋**(gold set)입니다.

| 주어 | 관계 | 목적어 | 원문 문장 번호 |
|---|---|---|---|
| 민수 | 빌림 | 파이썬 입문 | 1 |
| 민수 | 빌림 | SQL 첫걸음 | 1 |
| 지우 | 빌림 | 통계 기초 | 2 |
| 파이썬 입문 | 분류됨 | 프로그래밍 | 3 |
| SQL 첫걸음 | 분류됨 | 프로그래밍 | 3 |
| 통계 기초 | 분류됨 | 통계 | 4 |

골드는 다음 순서로 작성합니다.

1. 포함할 관계와 제외할 조건을 정합니다.  
2. 추출 결과를 보지 않고 원문을 읽습니다.  
3. 정답 관계를 출처와 함께 적습니다.

골드는 빌림 3관계와 분류됨 3관계로, 모두 6관계입니다.  
문장 번호는 앞의 원문 (1)~(7)과 같습니다.

- 문장 5: 대출 예정이므로 0건  
- 문장 6: 빌리지 않았으므로 0건  
- 문장 7: 추천은 허용 관계가 아니므로 0건

**0건도 검토 완료로 남깁니다.** 그래야 읽지 않은 문장과 구분할 수 있습니다.

### 1. 원문에서 찾은 정답 관계를 기록합니다

`demo_gold`에 원문을 읽고 확인한 정답 6개를 직접 적습니다.  
아래 코드는 정답을 자동으로 찾지 않습니다. 적어 둔 정답에 출처와 근거 문장을 붙입니다.

In [ ]:
# 원문에서 확인한 정답 6개를 기록하고 각각의 근거 문장을 붙입니다.

# demo_gold는 원문을 읽고 작성한 정답 관계 목록입니다. sentence_id는 근거 문장 번호입니다.
# 한 문장에 두 책이 나열되면 관계도 두 건으로 적습니다.
demo_gold = [
    {"subject": "민수", "relation": "빌림", "object": "파이썬 입문", "sentence_id": 1},
    {"subject": "민수", "relation": "빌림", "object": "SQL 첫걸음", "sentence_id": 1},
    {"subject": "지우", "relation": "빌림", "object": "통계 기초", "sentence_id": 2},
    {"subject": "파이썬 입문", "relation": "분류됨", "object": "프로그래밍", "sentence_id": 3},
    {"subject": "SQL 첫걸음", "relation": "분류됨", "object": "프로그래밍", "sentence_id": 3},
    {"subject": "통계 기초", "relation": "분류됨", "object": "통계", "sentence_id": 4},
]

# 각 정답에 출처 문서와 원문 문장을 붙여 나중에도 판정 근거를 확인할 수 있게 합니다.
for row in demo_gold:
    row["source_doc_id"] = "demo_library"

    # 문장 번호는 1부터, 파이썬 목록의 인덱스는 0부터이므로 1을 뺍니다.
    row["evidence"] = demo_sentences[row["sentence_id"] - 1]

print("기록한 정답 수:", len(demo_gold))
for row in demo_gold:
    print(f"문장 {row['sentence_id']}:", triple_key(row))

### 2. 모든 문장을 검토했는지 확인합니다

`demo_gold`에는 정답이 없는 문장 5~7이 나타나지 않습니다.  
이 문장도 읽었는지 확인하려고 **모든 문장의 검토 기록**인 `demo_checklist`를 작성합니다.  
문장 1을 읽고 정답 2개를 기록했다면 `reviewed=True, n_gold=2`로 적습니다.  
검토표에서 정답을 뽑는 것이 아니라, **정답을 작성한 뒤 빠뜨린 문장이 없는지 확인**하는 과정입니다.

In [ ]:
# 정답이 없는 문장까지 검토했는지 기록하고, 정답 목록과 개수가 맞는지 확인합니다.

# demo_checklist는 모든 원문 문장의 검토 기록입니다. 정답이 0건인 문장도 남깁니다.
# reviewed는 검토 완료 여부, n_gold는 해당 문장의 정답 관계 수, reason은 판정 이유입니다.
# n_gold는 demo_gold에 기록한 해당 문장의 관계 수와 같아야 합니다.
demo_checklist = [
    {"sentence_id": 1, "reviewed": True, "n_gold": 2, "reason": "실제 대출 두 권"},
    {"sentence_id": 2, "reviewed": True, "n_gold": 1, "reason": "실제 대출 한 권"},
    {"sentence_id": 3, "reviewed": True, "n_gold": 2, "reason": "두 책의 분야 분류"},
    {"sentence_id": 4, "reviewed": True, "n_gold": 1, "reason": "한 책의 분야 분류"},
    {"sentence_id": 5, "reviewed": True, "n_gold": 0, "reason": "대출 예정이므로 제외"},
    {"sentence_id": 6, "reviewed": True, "n_gold": 0, "reason": "빌리지 않았으므로 제외"},
    {"sentence_id": 7, "reviewed": True, "n_gold": 0, "reason": "추천은 허용 관계에 없으므로 제외"},
]

# 전체 문장 번호와 검토 완료 문장 번호를 비교해 빠진 문장이나 잘못 적은 번호를 찾습니다.
sentence_ids = set(range(1, len(demo_sentences) + 1))
reviewed_ids = {row["sentence_id"] for row in demo_checklist if row["reviewed"]}

print(f"검토 완료: {len(reviewed_ids)} / {len(sentence_ids)}문장")
print("검토에서 빠진 문장:", sorted(sentence_ids - reviewed_ids))
print("범위 밖에 적은 문장:", sorted(reviewed_ids - sentence_ids))

# 실제 정답 목록에서 문장별 관계 수를 세어 검토표에 적은 n_gold와 비교합니다.
gold_counts = Counter(row["sentence_id"] for row in demo_gold)
count_mismatches = [row["sentence_id"] for row in demo_checklist
                    if row["n_gold"] != gold_counts[row["sentence_id"]]]
print("검토표와 정답 수가 다른 문장:", count_mismatches)

### 아락실과립 원문에서 직접 골드를 작성합니다

이번 포함 기준은 **[이상반응] 절에 이름으로 적힌 증상**입니다.  
가능성 표현이 있어도 포함하고, 복용 중지·상담 같은 행동 지시는 제외합니다.  
도서관 기록과 포함 기준이 다릅니다. 자료마다 기준을 먼저 확인하세요.

In [ ]:
# 아락실과립의 이상반응 원문을 읽고 직접 골드를 작성할 준비를 합니다.

# fa_corpus.jsonl: 제품 8개의 원문과 절별 텍스트입니다. side_effect에는 [이상반응] 내용이 있습니다.
fa_docs = {row["doc_id"]: row for row in load_rows("fa_corpus.jsonl")}
annotation_doc = fa_docs["drug_198400661"]

print("제품:", annotation_doc["title"])
print("[이상반응]", annotation_doc["side_effect"])

### 🖐️ 함께 따라하기: 원문에서 정답 목록을 작성합니다

**할 일**: **my_gold**를 증상마다 딕셔너리 한 개가 있는 목록으로 만드세요.

| 칸 | 적을 내용 |
|---|---|
| subject | annotation_doc의 title |
| relation | HAS_SIDE_EFFECT(이상반응 관계) |
| object | 원문에 이름으로 적힌 증상 |
| source_doc_id | drug_198400661 |
| evidence_section | [이상반응] |
| evidence | annotation_doc의 side_effect |

도서관은 문장 번호로 출처를 찾았지만, 여기서는 **[이상반응] 절 전체**를 근거로 사용합니다.  
문서 ID와 절 이름, 원문을 함께 남기므로 `sentence_id`는 생략합니다.

write_rows로 my_gold.jsonl에 저장하세요. 다음 절에서 이 목록을 채점 기준으로 사용합니다.

**확인 기준**: 증상만 포함하고 중복·행동 지시를 제외합니다.

In [ ]:
# 🖐️ 함께 따라하기: 원문에서 정답 목록을 작성합니다

# 증상 이름을 먼저 고르고 각 항목에 제품명, 관계, 출처, 근거를 붙이세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

관계가 없는 문장은 검토 기록에서 빼도 될까요?

<details><summary>정답 보기</summary>

관계는 0건으로 적되 검토 기록은 남깁니다. 그래야 관계가 없는 문장과 읽지 않은 문장을 구분할 수 있습니다.

</details>

## 5. 정밀도·재현율·F1은 맞힘과 누락을 함께 보여 줍니다

### 5-1. 같은 관계를 한 번씩 비교합니다

여기서 한 항목은 **고유한 관계**이며 `(주어, 관계, 목적어)`로 비교합니다.  
세 문자열이 모두 같을 때만 정답과 일치한다고 셉니다. 이것을 **완전일치**(exact match)라고 합니다.  
같은 관계가 여러 번 있어도 정답을 여러 번 맞힌 것은 아니므로 집합으로 바꿉니다.  
출처·근거는 원래 목록에 보존합니다.

**골드는 그대로 두고, 평가할 추출 목록을 두 가지로 나누어 비교합니다.**  
먼저 스키마 통과 5건을 골드 6건과 비교하고, 이어서 두 검사 통과 4건도 같은 골드로 계산합니다.  
두 번째 목록에는 근거가 출처 원문에 그대로 있는 추출만 들어갑니다.

<img src="images/library_two_relation_evaluation.png" width="1000" alt="스키마 통과 추출 5관계와 고정 골드 6관계 비교">

### 정답 여부와 추출 여부로 네 경우를 구분합니다

**양성은 정답 관계, 음성은 정답이 아닌 관계를 뜻합니다.**  
추출기는 관계를 뽑으면 양성으로 예측한 것이고, 뽑지 않으면 음성으로 예측한 것입니다.

| 구분 | 원래 정의 | 이 추출 문제에서의 뜻 | 도서관 예시 |
|---|---|---|---|
| TP (True Positive, 참양성) | 실제 양성을 양성으로 예측 | 골드에 있는 관계를 추출함 | 3건. 민수 → 빌림 → 파이썬 입문 |
| FP (False Positive, 거짓양성) | 실제 음성을 양성으로 예측 | 골드에 없는 관계를 추출함 | 2건. 민수 → 빌림 → 머신러닝 실습 |
| FN (False Negative, 거짓음성) | 실제 양성을 음성으로 예측 | 골드에 있는 관계를 놓침 | 3건. 민수 → 빌림 → SQL 첫걸음 |
| TN (True Negative, 참음성) | 실제 음성을 음성으로 예측 | 정답이 아닌 관계를 추출하지 않음 | 후보 전체를 정하지 않았으므로 집계하지 않음 |

<img src="images/triple_confusion_matrix.png" width="1000" alt="행은 실제 정답, 열은 추출 결과인 혼동 행렬. TP3, FN3, FP2, TN은 미집계">

**위 그림과 아래 공식은 스키마 통과 추출 5건을 골드 6건과 비교한 결과입니다.**  
이 추출 5건에는 근거가 원문과 일치하지 않는 1건도 포함됩니다.  
골드는 앞에서 정한 포함 기준으로 작성한 정답 목록입니다. 표기가 다르면 완전일치에서는 다른 관계로 셉니다.

### 같은 맞힘 수를 두 가지 분모로 나눕니다

| 지표 | 무엇을 나누나요? | 공식 | 도서관 예시 |
|---|---|---|---|
| 정밀도 (precision) | 맞힌 3건 ÷ 뽑은 5건 | TP / (TP + FP) | 3 / 5 = 60% |
| 재현율 (recall) | 맞힌 3건 ÷ 정답 6건 | TP / (TP + FN) | 3 / 6 = 50% |
| F1 | 정밀도와 재현율의 조화평균 | 2 × TP / (2 × TP + FP + FN) | 6 / 11 ≈ 0.5455 |

**정밀도는 뽑은 관계를, 재현율은 찾아야 할 정답을 분모로 씁니다.**  
F1은 같은 추출과 같은 골드에서 구한 두 비율을 함께 보는 점수입니다.  
별도로 사람 판정한 3절의 샘플 정밀도를 이 F1에 섞지 않습니다.

### 골드가 있을 때와 없을 때의 평가 방법

**샘플 정밀도와 골드 정밀도는 모두 “추출한 관계 중 맞는 관계의 비율”이라는 같은 개념입니다.**  
샘플에서도 맞는 항목은 TP, 틀린 항목은 FP이므로 공식은 `TP / (TP + FP)`로 같습니다.

| 상황 | 평가 방법 |
|---|---|
| 골드가 없음 | 두 검사를 통과한 추출에서 표본을 뽑고, 사람이 판정해 정밀도를 추정 |
| 평가 범위를 충분히 포함하는 신뢰할 만한 골드가 있음 | 평가할 전체 추출을 골드와 비교해 정밀도·재현율·F1 계산 |
| 점수가 낮은 이유를 확인하고 싶음 | FP와 FN의 원문을 읽어 의미 오류, 표기 차이, 골드 누락을 확인 |

**충분한 골드가 있다면 정밀도를 구하기 위해 샘플 정밀도를 따로 계산할 필요는 없습니다.**  
이 교안에서는 두 평가 방법을 익히기 위해 모두 실습합니다.  
샘플은 사람의 의미 판정, 여기의 골드 평가는 문자열 완전일치를 사용하므로 값이 다를 수 있습니다.

In [ ]:
# 추출과 골드를 맞힘, 잘못 뽑음, 놓침으로 나누고 중복 관계의 처리도 확인합니다.

# &는 양쪽 모두, -는 왼쪽에만 있는 항목입니다. 빼는 순서에 주의합니다.
predicted_keys = {triple_key(row) for row in demo_predictions}
gold_keys = {triple_key(row) for row in demo_gold}

correct_keys = predicted_keys & gold_keys
print("맞힘:", sorted(correct_keys))

wrong_keys = predicted_keys - gold_keys
print("잘못 뽑음:", sorted(wrong_keys))

missed_keys = gold_keys - predicted_keys
print("놓침:", sorted(missed_keys))

# 같은 관계를 한 행 더 붙여도 고유 관계 수는 늘지 않습니다.
duplicate_demo = demo_predictions + [demo_predictions[0]]
print("스키마 통과 추출에 중복을 추가한 행 수:", len(duplicate_demo))
print("고유 관계 수:", len({triple_key(row) for row in duplicate_demo}))

In [ ]:
# TP, FP, FN 수를 공식에 넣어 정밀도, 재현율, F1을 직접 계산합니다.

tp, fp, fn = len(correct_keys), len(wrong_keys), len(missed_keys)

precision = tp / (tp + fp)
print(f"정밀도: {tp} / {tp + fp} = {precision:.2f}")

recall = tp / (tp + fn)
print(f"재현율: {tp} / {tp + fn} = {recall:.2f}")

f1 = 2 * tp / (2 * tp + fp + fn)
print(f"F1: {2 * tp} / {2 * tp + fp + fn} = {f1:.2f}")

In [ ]:
# 추출과 골드를 비교해 TP, FP, FN, 정밀도, 재현율, F1을 계산할 함수를 정의합니다.

def measure_exact(rows, gold_rows):
    """고유 관계의 완전일치 TP, FP, FN과 정밀도, 재현율, F1을 돌려줍니다."""
    # 집합으로 바꿔 같은 관계를 여러 번 뽑아도 한 번만 셉니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold_rows}
    tp = len(predicted & expected)  # 추출 결과와 골드 양쪽에 있는 관계입니다.
    fp = len(predicted - expected)  # 골드에 없는 추출입니다. 표기 차이도 포함합니다.
    fn = len(expected - predicted)  # 골드에는 있지만 추출하지 못한 관계입니다.
    # 분모가 없으면 0점 대신 미산출(None)로 남깁니다.
    precision = tp / len(predicted) if predicted else None
    recall = tp / len(expected) if expected else None

    # 골드가 있는데 아무것도 뽑지 않으면 F1은 0입니다. 골드가 없으면 평가에서 별도 표시합니다.
    f1 = 2 * tp / (2 * tp + fp + fn) if expected else None

    return {"predicted": len(predicted), "gold": len(expected), "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# 오답만 지워도 재현율이 그대로인지 원래 결과와 비교합니다.

demo_metrics = measure_exact(demo_predictions, demo_gold)
print("스키마 통과 추출:", demo_metrics)

demo_final_metrics = measure_exact(demo_verbatim, demo_gold)
print("스키마와 근거 원문 검사 통과 추출:", demo_final_metrics)
print("교정할 관계:", [triple_key(row) for row in demo_quote_review])

# 오답만 지웠다는 가정입니다. 실제 추출기를 개선한 결과는 아닙니다.
deletion_only = [row for row in demo_predictions if triple_key(row) not in wrong_keys]
deletion_metrics = measure_exact(deletion_only, demo_gold)
print("오답만 지운 경우:", deletion_metrics)

두 검사 통과 추출은 **TP 2, FP 2, FN 4**로 정밀도 50%, 재현율 33.33%, F1 0.4입니다.  
`통계 기초 → 분류됨 → 통계`는 맞는 관계지만, 근거 문구가 원문과 달라 두 검사 통과 목록에는 없습니다.  
**이 관계는 골드에 그대로 있으며, 두 검사 통과 목록을 평가할 때 FN으로 셉니다.**  
교정할 것은 골드가 아니라 추출된 트리플의 근거 문구입니다.

스키마 통과 목록에서 오답 두 관계를 지우면 정밀도는 1이 됩니다.  
그러나 놓친 빌림 2관계와 분류됨 1관계는 여전히 없어 재현율은 3/6입니다.  
**정밀도가 낮으면 FP, 재현율이 낮으면 FN을 먼저 읽습니다.**

| 낮은 지표 | 확인할 오류 | 교정 방향 |
|---|---|---|
| 정밀도 | LLM이 원문에 없거나 제외 대상인 관계를 추출함 | 잘못 뽑은 사례를 보여 주고, 어떤 관계를 제외할지 프롬프트에 명시 |
| 재현율 | LLM이 나열된 대상이나 앞뒤 문맥의 관계를 놓침 | 대상마다 관계를 확인하도록 지시하고, 필요한 앞뒤 문장을 함께 입력 |
| 재현율 | 맞는 관계를 뽑았지만 인용이나 타입 오류로 기각됨 | 기각 사유에 따라 해당 오류를 고친 뒤 같은 검사에 다시 통과시키기 |
| F1 | 정밀도 또는 재현율이 낮음 | 위 오류를 교정하고 두 지표를 다시 계산. 기존 정답이 사라졌는지도 확인 |

도서관의 SQL 첫걸음은 대출과 분류 모두 빠졌으므로 “**나열된 책마다 관계를 각각 추출하라**”고 지시할 수 있습니다.  
이름 표기만 달라 생긴 FP와 FN은 의미 오류와 구분해 기록합니다.  
개선 효과는 골드를 결과에 복사해 넣는 대신, 다시 추출한 결과로 평가합니다.

코드로 골드와의 차이를 찾은 뒤, 아래에서 FP와 FN의 근거를 각각 읽습니다.

In [ ]:
# 정밀도와 재현율을 낮춘 항목을 각각 원문 근거로 돌아가 확인합니다.

precision_problem = next(row for row in demo_predictions if triple_key(row) in wrong_keys)
print("정밀도를 낮춘 FP:", triple_key(precision_problem))
print("원문 근거:", precision_problem["evidence"])

recall_problem = next(row for row in demo_gold if triple_key(row) in missed_keys)
print("재현율을 낮춘 FN:", triple_key(recall_problem))
print("원문 근거:", recall_problem["evidence"])

In [ ]:
# 직접 쓴 my_gold와 비교할 아락실과립의 이상반응 트리플을 고릅니다.

# fa_candidates는 앞에서 스키마와 근거 원문 검사를 통과한 같은 제품 8개의 사용 후보입니다.
fa_all = fa_candidates
araksil_predictions = [row for row in fa_all if row["source_doc_id"] == "drug_198400661"
                       and row["relation"] == "HAS_SIDE_EFFECT"]

print("실제 추출:", [triple_key(row) for row in araksil_predictions])

### 🖐️ 함께 따라하기: 내 골드로 채점하고 실제 오류가 있는 자료로 넓힙니다

**할 일**  
- araksil_predictions와 my_gold를 평가해 **my_metrics**에 담으세요.  
- fa_gold.jsonl을 **fa_gold**로 읽으세요. 여덟 제품의 이상반응 정답입니다.  
- fa_all에서 문서 ID가 fa_docs에 있고 관계가 HAS_SIDE_EFFECT인 행만 **fa**에 담으세요.  
- fa와 fa_gold를 평가해 **fa_metrics**에 담고 두 점수를 출력하세요.

**확인 기준**: 직접 쓴 골드가 맞으면 아락실은 TP 3·FP 0·FN 0입니다.  
여덟 제품 전체는 TP 40·FP 7·FN 4입니다. 정밀도 분모는 47, 재현율 분모는 44입니다.  
02에서는 원본 프로젝트의 **목적어가 해당 절에 있는지 확인하는 검사**까지 통과한 42관계를 사용합니다.  
여기의 47관계와 검사 단계가 다릅니다.

제공 골드는 이상반응 절의 증상 이름을 세며 괄호 안 부연 증상·행동 지시는 별도로 세지 않습니다.  
빈 절은 관계 0건으로 기록합니다. 이것이 의학적으로 이상반응이 없다는 뜻은 아닙니다.

In [ ]:
# 🖐️ 함께 따라하기: 내 골드로 채점하고 실제 오류가 있는 자료로 넓힙니다

# 추출과 골드의 문서, 관계 범위를 맞춘 뒤 measure_exact를 사용하세요.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

TP와 골드는 같은데 FP가 줄었습니다. 정밀도와 재현율은 어떻게 달라지나요?

<details><summary>정답 보기</summary>

FP가 줄면 정밀도는 높아집니다. 골드와 TP가 같으므로 재현율은 그대로입니다.

</details>

### 🖐️ 함께 따라하기: 실제 제품 추출의 FP와 FN 목록을 만듭니다

**할 일**  
- fa와 fa_gold를 고유 관계 집합으로 바꾸고 **fa_fp**, **fa_fn**을 구하세요.  
- 각 목록의 관계와 개수를 출력하세요.  
- **fa_summary**에 metrics, fp, fn을 담아 my_product_metrics.json으로 저장하세요. 집합은 정렬한 목록으로 저장합니다.

**확인 기준**: FP 7관계·FN 4관계입니다. 골드와 다르다는 계산 결과까지 기록합니다.  
차이가 난 이유는 02에서 원문을 읽고 조사합니다.

In [ ]:
# 🖐️ 함께 따라하기: 실제 제품 추출의 FP와 FN 목록을 만듭니다

# FP는 추출에서 골드를 빼고, FN은 골드에서 추출을 뺍니다.
# 여기에 코드를 작성하세요.

### ✅ 바로 확인 퀴즈

완전일치에서 FP로 표시된 항목은 모두 원문을 틀리게 이해한 결과인가요?

<details><summary>정답 보기</summary>

아닙니다. 같은 개체라도 괄호·대소문자 등 표기가 다르면 FP와 FN이 생깁니다. 원문과 주석 기준을 읽어 의미 오류와 표기 차이를 구분합니다.

</details>

## 🚀 종합 클론코딩: 품질 지표를 하나씩 계산하기

아래 준비 코드를 실행한 뒤, **각 코드 블록의 주석을 따라 직접 계산하세요.**  
스키마 준수율, 근거 원문 일치율, 정밀도, 재현율, F1, 샘플 정밀도를 순서대로 출력합니다.

**골드는 같은 논문 6편의 정답 68관계이며, 모든 비교에서 그대로 사용합니다.**  
**검사 전 42행은 추출 자체의 품질, 두 검사 통과 17행은 실제 사용할 결과의 품질을 평가합니다.**  
이 저장본은 스키마를 42행 모두 통과합니다. 스키마 기각이 있어도 첫 비교에는 포함합니다.  
**근거 원문 검사**는 `evidence`가 비어 있지 않고 출처 원문에 그대로 포함되는지 확인하는 것입니다.  
이 검사는 관계가 정답인지를 판정하지 않습니다.

**표본은 스키마와 근거 원문 검사를 모두 통과한 목록에서만 뽑습니다.**  
마지막 셀에는 원문과 [포함·제외 기준](data/gold_guideline.json)을 대조한 **샘플별 판정 예시**가 있습니다.  
점수와 이유를 확인하고, 샘플 정밀도 계산식은 직접 작성하세요.

**확인 기준**

| 지표 | 검사 전 추출 | 두 검사 통과 추출 |
|---|---|---|
| 비교할 골드 | 68관계 전체 | 같은 68관계 전체 |
| 스키마 준수율 | 42 / 42 = 100% | - |
| 근거 원문 일치율 | 17 / 42 ≈ 40.48% | - |
| TP / FP / FN | 31 / 11 / 37 | 11 / 6 / 57 |
| 정밀도 | 73.81% | 64.71% |
| 재현율 | 45.59% | 16.18% |
| F1 | 0.5636 | 0.2588 |
| 검토 표본 | - | 통과 17행에서 10행 선정 |

**근거 원문 검사에 실패한 추출 25행은 `paper_quote_review`에 따로 남깁니다.**  
그중 20관계는 골드와 일치하지만, 두 검사 통과 목록에는 없어 이 목록을 평가할 때 FN으로 셉니다.  
**골드를 고치는 것이 아니라, 해당 추출의 원문과 근거를 확인해야 합니다.**

In [ ]:
# 논문 원문, 추출된 트리플과 정답 목록을 읽습니다.

import json
import random
from pathlib import Path

data_dir = Path("data")

# kg_corpus.jsonl: 논문 6편의 원문입니다. 출처 ID로 원문을 찾도록 사전으로 읽습니다.
paper_lines = (data_dir / "kg_corpus.jsonl").read_text(encoding="utf-8").splitlines()
paper_doc_rows = [json.loads(line) for line in paper_lines if line.strip()]
paper_docs = {row["doc_id"]: row for row in paper_doc_rows}

# triples_raw.jsonl: 예시를 추가하기 전에 추출된 논문 트리플 42행입니다. 검사 대상입니다.
paper_lines = (data_dir / "triples_raw.jsonl").read_text(encoding="utf-8").splitlines()
paper_raw = [json.loads(line) for line in paper_lines if line.strip()]

# gold_triples.jsonl: 같은 논문 6편의 정답 68관계입니다. 전체를 고정된 정답 기준으로 사용합니다.
paper_lines = (data_dir / "gold_triples.jsonl").read_text(encoding="utf-8").splitlines()
paper_gold = [json.loads(line) for line in paper_lines if line.strip()]

# 논문에서 허용하는 관계와 주어, 목적어 타입입니다. 앞의 의약품 규칙과 구분합니다.
paper_signatures = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

print("원문 문서:", len(paper_docs), "/ 추출:", len(paper_raw), "/ 골드:", len(paper_gold))

In [ ]:
# 1. 스키마 준수율을 계산하세요.

# paper_raw에서 source_doc_id가 paper_docs에 있는 행만 paper_scoped에 담으세요.
# for문으로 허용 관계인지 먼저 확인하고, 주어 타입과 목적어 타입을 각각 검사하세요.
# 세 조건을 모두 통과한 행만 paper_valid에 담으세요.
# paper_schema_rate = 통과 행 수 / 검사한 행 수입니다.
# 통과 수, 검사 수와 준수율을 출력하세요. 기각 행도 분모에 포함합니다.

In [ ]:
# 2. 근거 원문 일치율을 계산하세요.

# paper_valid에서 evidence가 비어 있지 않고 출처 원문의 text에 그대로 있는 행만 고르세요.
# for문 안에서 빈 근거인지 먼저 확인하고, 원문에 포함되는지 다음으로 검사하세요.
# 두 검사를 통과한 추출은 paper_candidates, 근거가 비었거나 원문과 다른 추출은 paper_quote_review에 담으세요.
# 원문은 paper_docs[row["source_doc_id"]]["text"]에서 확인합니다.
# paper_verbatim_rate = len(paper_candidates) / len(paper_valid)입니다.
# 원문 일치 수, 검사 수와 일치율을 출력하세요. 원본 목록은 수정하지 않습니다.

In [ ]:
# 3. 고유 관계를 비교해 TP, FP, FN을 구하세요.

# 비교 키는 (subject, relation, object)입니다. 같은 키는 한 번만 세도록 집합으로 만드세요.
# paper_gold_keys에는 paper_gold의 모든 관계를 넣으세요. 골드를 제외하거나 수정하지 않습니다.
# 추출 자체의 품질은 두 검사 전의 paper_scoped, 최종 사용 결과의 품질은 paper_candidates로 평가합니다.
# 여기서는 스키마와 근거 원문 검사를 모두 통과한 트리플만 실제로 사용한다고 정합니다.
# paper_before_keys: paper_scoped의 관계 집합, paper_after_keys: paper_candidates의 관계 집합입니다.
# 각 집합은 set()으로 만들고, for문 안에서 관계 키를 add로 넣으세요.
# 각 단계의 중복 행 수는 목록 길이에서 관계 집합의 길이를 빼서 출력하세요.

# TP는 추출과 골드의 교집합, FP는 추출에만 있는 관계, FN은 골드에만 있는 관계의 수입니다.
# 검사 전 추출은 paper_before_tp, paper_before_fp, paper_before_fn으로 계산해 출력하세요.
# 두 검사 통과 추출은 paper_tp, paper_fp, paper_fn으로 계산해 출력하세요. 같은 paper_gold_keys를 사용합니다.

# paper_excluded_tp_keys에 검사 전 TP 중 두 검사 통과 목록에 없는 관계를 담으세요.
# 이 관계는 골드에 그대로 있으므로 최종 결과에서는 FN입니다. 개수를 출력하세요.

In [ ]:
# 4. 정밀도 = TP / (TP + FP)를 계산하세요.

# 검사 전 추출은 paper_before_precision, 두 검사 통과 추출은 paper_precision에 담으세요.
# 각 값을 계산한 바로 다음 줄에서 백분율로 출력하세요.

In [ ]:
# 5. 재현율 = TP / (TP + FN)을 계산하세요.

# 검사 전 추출은 paper_before_recall, 두 검사 통과 추출은 paper_recall에 담으세요.
# 두 계산에서 골드의 수는 같습니다. 각 값을 바로 백분율로 출력하세요.

In [ ]:
# 6. F1 = 2 * TP / (2 * TP + FP + FN)을 계산하세요.

# 검사 전 추출은 paper_before_f1, 두 검사 통과 추출은 paper_f1에 담으세요.
# 각각 소수 넷째 자리까지 출력하세요. 서로 다른 단계의 TP, FP, FN을 섞지 않습니다.

In [ ]:
# 7. 두 검사를 통과한 목록에서 표본을 뽑으세요.

# paper_candidates에서만 random.Random(42).sample로 최대 10행을 뽑아 paper_sample에 담으세요.
# 각 표본의 번호, 트리플, evidence와 source_doc_id를 출력하세요.
# 첫 표본의 전체 원문은 paper_docs[paper_sample[0]["source_doc_id"]]["text"]에서 확인합니다.

# 다음 셀의 paper_scores는 이 데이터와 시드 42로 뽑은 표본 순서의 판정입니다.
# 표본을 바꾸면 판정도 해당 표본에 맞게 다시 기록해야 합니다.

In [ ]:
# 앞서 출력한 고정 표본 10건을 원문과 포함 기준으로 대조한 판정 예시입니다.
# BINDS에는 표적에 대한 작용, 대사 효소와 수송체 관계가 포함됩니다.
# 0은 이 원문과 기준으로 관계를 채택하기 어렵다는 뜻입니다.
paper_scores = [
    0,  # 1. tacrolimus: CYP3A5 관련 권고만 언급하고 작용이나 대사 역할은 설명하지 않음.
    1,  # 2. tramadol: 앞뒤 문장에서 CYP2D6 활성과 활성 대사산물 생성이 연결됨.
    1,  # 3. Carbidopa: AHR을 활성화한다고 보고함.
    0,  # 4. quetiapine: CYP3A4 관련 권고만 언급하고 작용이나 대사 역할은 설명하지 않음.
    0,  # 5. cannabinoids: PSD에 대한 근거가 제한적이며 완화 효과를 확정하지 않음.
    1,  # 6. Carbidopa: Parkinson disease 치료에 사용한다고 명시함.
    1,  # 7. clopidogrel: 앞뒤 문장에서 CYP2C19 활성 저하와 약물 활성화 감소가 연결됨.
    1,  # 8. simvastatin: 앞뒤 문장에서 SLCO1B1 수송 기능 저하와 약물 농도 증가가 연결됨.
    1,  # 9. Laquinimod: AHR 작용제라고 명시함.
    0,  # 10. Laquinimod: inflammatory bowel disease 치료를 위한 연구 관심만 언급함.
]

# 8. 위의 샘플별 점수로 샘플 정밀도를 계산하세요.

# paper_sample_correct에 1점인 표본 수를 담으세요.
# paper_sample_precision = paper_sample_correct / len(paper_sample)로 계산하고 분자, 분모, 백분율을 출력하세요.
# 골드 일치 여부를 사람 판정값으로 대신 사용하지 않습니다.